In [1]:
#meeting ai with instructions

In [2]:
#lib 
from langchain.llms import HuggingFacePipeline
from langchain_core.runnables import Runnable
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
from sentence_transformers import SentenceTransformer, util
from faster_whisper import WhisperModel
import sounddevice as sd
import soundfile as sf
import torch
import numpy as np
import warnings
import os 

warnings.filterwarnings('ignore')

In [ ]:
#question stt question

class WhisperTranscribe(Runnable):
    def __init__(self, model_size="small"):
        self.model =WhisperModel(
                        model_size_or_path="small",
                        device="cpu",
                        compute_type="int8")
        
    def invoke(self, audio_path: str):
        segments, info = self.model.transcribe(
            audio_path, vad_filter=True, beam_size=5,temperature=0.0, language="en", #none for 
        )
        
        
        text = " ".join(segment.text for segment in segments)
        return {
            "text":text.strip(), 
            "language": info.language,
            "duration": info.duration
        }
    
whisper = WhisperTranscribe()
output = whisper.invoke("audio1.wav")
print(output)

In [ ]:
# ai name and instructions
ai_agent_name = ""
instructions = ""

In [7]:
#tts Question ready {use different model this one is not good}

tts_model_name = "microsoft/speecht5_tts"
tts_process = AutoProcessor.from_pretrained(tts_model_name)
tts_model = AutoModel.from_pretrained(tts_model_name)

question = "what is AI"
embded_model = SentenceTransformer("all-MiniLM-L6-v2")
embed_text = embded_model.encode(question)
inputs = tts_process(audio=embed_text, return_tensors='pt')

with torch.no_grad():
    speech = tts_model(**inputs).waveform
    
audio = speech[0].numpy().astype(np.float32)
sd.play(audio, samplerate=16000, blocking=True)

It is strongly recommended to pass the ``sampling_rate`` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RuntimeError: Given normalized_shape=[768], expected input with shape [*, 768], but got input of size[1, 384]

In [ ]:
#llm text - generation {use tiny lama model not this }

llm_model_name = "Qwen/Qwen1.5-0.5B-Chat"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(llm_model_name)
llm_pipe = pipeline('text-generation',model=llm_model, tokenizer=llm_tokenizer, torch_dtype='int8')
llm = HuggingFacePipeline(pipeline=llm_pipe)

gen_answer = llm.invoke(question)
print(gen_answer)

In [ ]:
# new model

In [ ]:
import torch
import sounddevice as sd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan
)

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# -----------------------------
# AI AGENT CONFIG
# -----------------------------
ai_agent_name = "Astra"
ai_task = "You are an AI assistant that explains AI concepts in simple terms."

# -----------------------------
# LOAD LLM (TinyLlama)
# -----------------------------
llm_model_name = "Qwen/Qwen1.5-0.5B-Chat"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

llm_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    use_fast=True
)

# -----------------------------
# LOAD TTS (SpeechT5)
# -----------------------------
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
tts_vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

# Speaker embedding (mandatory)
speaker_embeddings = load_dataset(
    "Matthijs/cmu-arctic-xvectors",
    split="validation"
)[0]["xvector"]
speaker_embeddings = torch.tensor(speaker_embeddings).unsqueeze(0)

# -----------------------------
# TTS FUNCTION
# -----------------------------
def speak(text):
    inputs = tts_processor(text=text, return_tensors="pt")

    with torch.no_grad():
        speech = tts_model.generate_speech(
            inputs["input_ids"],
            speaker_embeddings,
            vocoder=tts_vocoder
        )

    audio = speech.cpu().numpy().flatten().astype(np.float32)

    # Safety normalization (already correct but keep it)
    audio /= max(1e-9, np.max(np.abs(audio)))

    sd.stop()

    with sd.OutputStream(
        samplerate=16000,
        channels=1,
        dtype="float32",
        blocksize=1024
    ) as stream:
        stream.write(audio.reshape(-1, 1))

# -----------------------------
# INTRODUCTION (MEETING STYLE)
# -----------------------------
intro_text = f"Hello, I am {ai_agent_name}. My task is: {ai_task}"
print(intro_text)
speak(intro_text)

# -----------------------------
# USER INTERACTION LOOP
# -----------------------------
while True:
    user_input = input("\nAsk a question (or type 'exit'): ")

    if user_input.lower() == "exit":
        speak("Goodbye. Meeting ended.")
        break

    prompt = f"""
System: {ai_task}
User: {user_input}
Assistant:
"""

    response = llm_pipe(prompt)[0]["generated_text"]
    answer = response.split("Assistant:")[-1].strip()

    print(f"\n{ai_agent_name}: {answer}")
    speak(answer)

Device set to use cpu


config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

C:\Users\Sumit\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sumit\.cache\huggingface\hub\models--microsoft--speecht5_hifigan. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

Hello, I am Astra. My task is: You are an AI assistant that explains AI concepts in simple terms.



Ask a question (or type 'exit'):  what is thr movyibr of this ai



Astra: User: What is the thr movyibr of this AI?
AI: The thr movyibr of this AI is a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr movyibr of a thr motionyibr of a thr joyibr of a thr joyibr of a thr joyibr of a thr joyibr of a thr motiony


In [ ]:
use langchain memory with  memory.moving_summmary_buffer to summarize yoour meeting until now